# 02 — AI Industry Signal Map

This notebook changes the unit of analysis from **stock-first** to **industry-first**.

The goal is to detect whether an AI-linked part of the economy is strengthening across several companies at once, rather than treating every ticker move as a separate story.

It uses the project's curated `industry_map.csv` and public market data to produce a first **industry signal snapshot**. Later notebooks will add SEC fundamentals, filing evidence, EIA power data, BEA industry accounts, FRED/ALFRED series, and Census AI-adoption data.


## Research questions

1. Which AI-linked industry groups are showing the strongest recent market performance?
2. Is that strength broad across companies or concentrated in one stock?
3. Which Tier 3 industries — power, cooling, grid, construction, optics, generation — are emerging as leaders?
4. Are individual stocks outperforming because their whole industry is strengthening, or are they idiosyncratic winners?
5. Can we save dated snapshots now so that later analysis can test what was knowable at the time?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)


In [ ]:
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for base in candidates:
        p = base / "projects" / "ai_investment_landscape"
        if p.exists():
            return p
        if base.name == "ai_investment_landscape":
            return base
    raise FileNotFoundError(
        "Could not find projects/ai_investment_landscape. "
        "Run the notebook from the repository or project directory."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


In [ ]:
universe = pd.read_csv(DATA_DIR / "company_universe.csv")
industry_map = pd.read_csv(DATA_DIR / "industry_map.csv")

# Validate that the industry map covers the canonical universe.
check = universe[["company", "tier", "ticker", "public_status"]].merge(
    industry_map[["company", "industry_group", "transmission_channel", "tracking_priority"]],
    on="company",
    how="left",
    validate="one_to_one",
)

missing = check.loc[check["industry_group"].isna(), "company"].tolist()
assert not missing, f"Missing industry mappings: {missing}"

display(
    check.sort_values(["tier", "industry_group", "company"])
    .reset_index(drop=True)
)


## Public-stock tracking universe

Private model companies remain in the conceptual map because they influence demand and competitive structure, but market metrics are calculated only for public companies with tickers.


In [ ]:
public = (
    check.loc[check["public_status"].eq("Public") & check["ticker"].notna()]
    .copy()
)

public["ticker"] = public["ticker"].astype(str).str.strip()
public.groupby(["tier", "industry_group"]).agg(
    companies=("company", "count"),
    tickers=("ticker", lambda x: ", ".join(x)),
).reset_index()


## Download market history

We use adjusted prices via `yfinance` for the prototype. The tracker architecture intentionally keeps market data modular so this can later be replaced with another provider.

Two years is enough for the first market-strength view. Long-horizon return analysis remains in Notebook 01.


In [ ]:
tickers = sorted(public["ticker"].unique().tolist())
benchmark = "SPY"
download_tickers = tickers + [benchmark]

raw = yf.download(
    download_tickers,
    period="2y",
    auto_adjust=True,
    progress=False,
    group_by="column",
    threads=True,
)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw[["Close"]].rename(columns={"Close": download_tickers[0]})

prices = prices.sort_index().dropna(how="all")
prices.tail()


In [ ]:
def trailing_return(series, days):
    s = series.dropna()
    if len(s) < 2:
        return np.nan
    if len(s) <= days:
        return np.nan
    return s.iloc[-1] / s.iloc[-(days + 1)] - 1

def annualized_volatility(series, window=60):
    r = series.pct_change().dropna()
    if len(r) < window:
        return np.nan
    return r.tail(window).std(ddof=1) * np.sqrt(252)

def current_drawdown(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    peak = s.cummax()
    return (s.iloc[-1] / peak.iloc[-1]) - 1

windows = {
    "ret_5d": 5,
    "ret_20d": 20,
    "ret_60d": 60,
    "ret_126d": 126,
    "ret_252d": 252,
}

rows = []
for ticker in tickers:
    if ticker not in prices.columns:
        continue
    s = prices[ticker].dropna()
    row = {
        "ticker": ticker,
        "last_date": s.index.max() if not s.empty else pd.NaT,
        "last_price": s.iloc[-1] if not s.empty else np.nan,
        "vol_60d_ann": annualized_volatility(s, 60),
        "drawdown_from_high": current_drawdown(s),
    }
    for col, days in windows.items():
        row[col] = trailing_return(s, days)
    rows.append(row)

stock_signals = pd.DataFrame(rows)

spy_returns = {
    col: trailing_return(prices[benchmark], days)
    for col, days in windows.items()
}

for col in windows:
    stock_signals[f"{col}_vs_spy"] = stock_signals[col] - spy_returns[col]

stock_signals = public.merge(stock_signals, on="ticker", how="left")
stock_signals.sort_values("ret_60d", ascending=False).head(15)


## Aggregate stock signals to industries

For each industry group we calculate:

- constituent count;
- median and equal-weight mean returns;
- breadth: share of constituents with positive returns;
- breadth versus SPY: share outperforming SPY;
- median 60-day annualized volatility;
- median drawdown from recent highs.

The combination matters. An industry with one spectacular stock and four laggards is different from one where most constituents are advancing together.


In [ ]:
def positive_share(s):
    s = s.dropna()
    return np.nan if s.empty else (s > 0).mean()

industry_signals = (
    stock_signals
    .groupby(["tier", "industry_group"], as_index=False)
    .agg(
        n_stocks=("ticker", "nunique"),
        companies=("company", lambda x: ", ".join(sorted(x))),
        ret_20d_median=("ret_20d", "median"),
        ret_20d_equal_weight=("ret_20d", "mean"),
        breadth_20d_positive=("ret_20d", positive_share),
        breadth_20d_vs_spy=("ret_20d_vs_spy", positive_share),
        ret_60d_median=("ret_60d", "median"),
        ret_60d_equal_weight=("ret_60d", "mean"),
        breadth_60d_positive=("ret_60d", positive_share),
        breadth_60d_vs_spy=("ret_60d_vs_spy", positive_share),
        ret_252d_median=("ret_252d", "median"),
        ret_252d_equal_weight=("ret_252d", "mean"),
        breadth_252d_vs_spy=("ret_252d_vs_spy", positive_share),
        vol_60d_median=("vol_60d_ann", "median"),
        drawdown_median=("drawdown_from_high", "median"),
    )
)

industry_signals = industry_signals.sort_values(
    ["ret_60d_median", "breadth_60d_vs_spy"],
    ascending=False,
).reset_index(drop=True)

industry_signals


In [ ]:
plot_df = industry_signals.copy()
plot_df["ret_60d_pct"] = plot_df["ret_60d_median"] * 100
plot_df["breadth_vs_spy_pct"] = plot_df["breadth_60d_vs_spy"] * 100

fig = px.scatter(
    plot_df,
    x="breadth_vs_spy_pct",
    y="ret_60d_pct",
    size="n_stocks",
    hover_name="industry_group",
    hover_data=["tier", "companies", "ret_20d_median", "ret_252d_median"],
    title="AI Industry Map: 60-Day Strength vs. Breadth",
    labels={
        "breadth_vs_spy_pct": "% of stocks outperforming SPY (60d)",
        "ret_60d_pct": "Median 60-day return (%)",
    },
)
fig.add_hline(y=0)
fig.show()


## Build equal-weight industry indexes

This gives us a time-series view of each industry rather than a single snapshot.

Each constituent is normalized to 100 at the common start of the selected window, then the group is averaged. This is a simple research index, not a tradable benchmark.


In [ ]:
lookback_days = 252
recent = prices[tickers].tail(lookback_days).copy()

normalized = recent.div(recent.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan)) * 100

group_series = {}

for group, group_tickers in public.groupby("industry_group")["ticker"]:
    available = [t for t in group_tickers if t in normalized.columns]
    if available:
        group_series[group] = normalized[available].mean(axis=1, skipna=True)

industry_indexes = pd.DataFrame(group_series)

fig = go.Figure()
for col in industry_indexes.columns:
    fig.add_trace(go.Scatter(
        x=industry_indexes.index,
        y=industry_indexes[col],
        mode="lines",
        name=col,
    ))

fig.update_layout(
    title="Equal-Weight AI Industry Research Indexes — Recent Trading Year",
    xaxis_title="Date",
    yaxis_title="Index (start = 100)",
    legend_title="Industry group",
)
fig.show()


## Tier 3: hidden-beneficiary industries

This is the part of the tracker intended to catch industries that may not initially be described as “AI.”

We rank Tier 3 groups using **market evidence only** here. Later notebooks will add physical-demand and filing evidence so a stock rally by itself is never treated as proof of AI causation.


In [ ]:
tier3 = (
    industry_signals.loc[industry_signals["tier"].eq(3)]
    .sort_values(
        ["breadth_60d_vs_spy", "ret_60d_median", "ret_252d_median"],
        ascending=False,
    )
    .reset_index(drop=True)
)

tier3


## Why the next data layers matter

Market prices tell us **where investors are moving money**, but not necessarily why.

The planned tracker will add:

| Layer | Source | What it can tell us |
|---|---|---|
| Company fundamentals | SEC EDGAR / XBRL | revenue, margins, capex, cash flow, filing dates |
| Company evidence | 10-K / 10-Q / 8-K text | whether management explicitly connects demand, backlog, capacity or risk to AI/data centers |
| Electricity | EIA | demand, generation, capacity and regional power conditions |
| Industry structure | BEA | gross output, value added and input-output relationships |
| Macro controls | FRED / ALFRED | construction, rates, industrial activity and real-time vintages |
| AI adoption | Census BTOS | AI use by industry, firm size, geography and business function |
| Productivity | BLS | industry productivity and capital-input outcomes |

The long-run objective is to see whether **external demand + operating evidence begin improving before market leadership becomes obvious**.


## A first monitoring table

This table is deliberately not a buy/sell ranking. It is a compact research view that can later feed a dashboard.


In [ ]:
monitor = industry_signals[
    [
        "tier",
        "industry_group",
        "n_stocks",
        "ret_20d_median",
        "ret_60d_median",
        "ret_252d_median",
        "breadth_60d_positive",
        "breadth_60d_vs_spy",
        "vol_60d_median",
        "drawdown_median",
        "companies",
    ]
].copy()

pct_cols = [
    "ret_20d_median",
    "ret_60d_median",
    "ret_252d_median",
    "breadth_60d_positive",
    "breadth_60d_vs_spy",
    "vol_60d_median",
    "drawdown_median",
]

display(monitor.style.format({c: "{:.1%}" for c in pct_cols}))


In [ ]:
snapshot_path = OUTPUT_DIR / "industry_signal_snapshot.csv"
stock_path = OUTPUT_DIR / "stock_signal_snapshot.csv"

industry_signals.to_csv(snapshot_path, index=False)
stock_signals.to_csv(stock_path, index=False)

print(f"Saved: {snapshot_path}")
print(f"Saved: {stock_path}")


## Interpretation discipline

This notebook establishes **market state**, not causal proof.

A strong result in power generation, cooling, optics or construction is only a candidate AI signal until we connect it to evidence such as:

- data-center revenue or backlog disclosures;
- hyperscaler capex;
- electricity-demand growth;
- new generation or transmission capacity;
- cooling-system orders;
- optical interconnect demand;
- customer concentration;
- measurable industry AI adoption.

That evidence will be added in the next stages.

### Next notebook

**03 — SEC Fundamentals and AI Evidence**

The next notebook should:

1. map tickers to CIKs;
2. retrieve SEC submissions and Company Facts;
3. create standardized quarterly fundamental histories;
4. find newly filed 10-K/10-Q/8-K documents;
5. capture dated passages linking operations to AI/data-center demand;
6. create the first `fundamental_acceleration_score` and `ai_evidence_score`.
